In [36]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *

In [37]:
model = "Llama-3-8B-Instruct_5_shot"

### Model Output to nice JSON and Failure 

In [38]:
input_dir = f"model_output/{model}/output/"
output_dir = f"model_output/{model}/formatted/"
failure_dir = f"model_output/{model}/failed/"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
if not os.path.exists(failure_dir):
    os.makedirs(failure_dir)

for filename in os.listdir(input_dir):
    if filename.endswith('.json'):
        input_file_path = os.path.join(input_dir, filename)
        output_file_path = os.path.join(output_dir, filename)
        failure_file_path = os.path.join(failure_dir, filename)
        try:
            file = read_json(input_file_path)
            print(f"Processing file: {filename}")
            save_json_to_file(file, output_file_path)
        except Exception as e:
            print(f"Error processing file {filename}: {e}")
            shutil.move(input_file_path, failure_file_path)

Processing file: Llama-3-8B-Instruct_NCT00050349_exc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00050349_inc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00061308_exc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00061308_inc_5_shot.json
Error processing file Llama-3-8B-Instruct_NCT00094861_exc_5_shot.json: Unterminated string starting at: line 1 column 1 (char 0)
Error processing file Llama-3-8B-Instruct_NCT00094861_inc_5_shot.json: Unterminated string starting at: line 1 column 1 (char 0)
Processing file: Llama-3-8B-Instruct_NCT00122070_exc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00122070_inc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00182520_exc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00182520_inc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00183885_exc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00183885_inc_5_shot.json
Processing file: Llama-3-8B-Instruct_NCT00198913_exc_5_shot.json
Processing file: Llama-3-8B

In [39]:
p2_label_path = "chia_label/p2"
ready_path = f"model_output/{model}/ready"
failed_model_path = f"model_output/{model}/failed_inner"
p2_model_formatted_path = f"model_output/{model}/formatted"
eval_path = f"evaluate/batch1"

for path in [ready_path, failed_model_path, p2_model_formatted_path]: 
    os.makedirs(path, exist_ok=True)

In [40]:
def extract_nct_number(filename):
    parts = filename.split('_')
    nct_number = None
    file_type = None
    for part in parts:
        if part.startswith("NCT"):
            nct_number = part
        if part in ["inc", "exc"]:
            file_type = part
    return nct_number+"_"+file_type


def extract_logical_structure(data):
    structure = defaultdict(int)
    def traverse(node, depth=0):
        nonlocal structure
        structure["depth"] = max(structure["depth"], depth)

        if isinstance(node, dict):
            for key in node:
                if key in ["AND", "OR", "NOT"]:
                    structure[key] += 1
                traverse(node[key], depth + 1)
        elif isinstance(node, list):
            for item in node:
                traverse(item, depth + 1)

    traverse(data)
    return dict(structure)

In [41]:
label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}

common_ncts = set(label_files.keys()).intersection(model_files.keys())

In [42]:
labels = []
predictions = []
success_data = []

In [43]:
for nct in common_ncts:# ["NCT00050349_exc"]
    try:
        label_data = read_json(label_files[nct])
        model_data = read_json(model_files[nct])
        label_structure = extract_logical_structure(label_data)
        model_structure = extract_logical_structure(model_data)
        success_data.append({
            'NCT': nct,
            'label_AND': label_structure.get('AND', 0),
            'label_OR': label_structure.get('OR', 0),
            'label_NOT': label_structure.get('NOT', 0),
            'label_DEPTH': label_structure.get('depth', 0),
            'model_AND': model_structure.get('AND', 0),
            'model_OR': model_structure.get('OR', 0),
            'model_NOT': model_structure.get('NOT', 0),
            'model_DEPTH': model_structure.get('depth', 0),
            'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
            'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
            'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
            'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0
            
        })

        labels.append(label_structure)
        predictions.append(model_structure)
        shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
    except Exception as e:
        print(f"Error processing NCT {nct}: {e}")
        shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))

Error processing NCT NCT01857167_exc: Extra data: line 58 column 5 (char 1533)
Error processing NCT NCT01214096_exc: Expecting ',' delimiter: line 69 column 2 (char 1904)
Error processing NCT NCT02557412_exc: Extra data: line 32 column 1 (char 783)
Error processing NCT NCT02053246_inc: Extra data: line 47 column 1 (char 1125)
Error processing NCT NCT02431442_inc: Expecting ',' delimiter: line 64 column 2 (char 2264)
Error processing NCT NCT02393287_inc: Extra data: line 53 column 1 (char 1230)
Error processing NCT NCT01082549_inc: Expecting ',' delimiter: line 33 column 13 (char 1215)
Error processing NCT NCT02195024_inc: Expecting ',' delimiter: line 80 column 2 (char 2427)
Error processing NCT NCT01803828_inc: Invalid control character at: line 11 column 29 (char 211)
Error processing NCT NCT01483118_inc: Expecting ',' delimiter: line 37 column 101 (char 1399)
Error processing NCT NCT00061308_exc: Extra data: line 37 column 4 (char 661)
Error processing NCT NCT00183885_exc: Expecting

In [44]:
df_success = pd.DataFrame(success_data).set_index('NCT')
df_success.to_csv(eval_path+f'/{model}_eval.csv')

df_success

,label_AND,label_OR,label_NOT,label_DEPTH,model_AND,model_OR,model_NOT,model_DEPTH,diff_AND,diff_OR,diff_NOT,diff_DEPTH
NCT,,,,,,,,,,,,
NCT02042287_inc,9,0,0,19,7,4,4,13,-1,1,1,-1
NCT00440245_inc,1,1,0,5,0,1,0,3,-1,0,0,-1
NCT01963754_inc,8,1,1,17,8,2,0,17,0,1,-1,0
NCT02394158_inc,4,0,0,9,2,2,0,9,-1,1,0,0
NCT02303171_exc,3,0,0,7,2,2,0,7,-1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
NCT02579733_inc,4,0,1,7,5,1,0,7,1,1,-1,0
NCT01424020_exc,4,1,0,9,1,3,0,9,-1,1,0,0
NCT02537899_inc,5,1,0,11,6,2,0,11,1,1,0,0


In [45]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    diffs = y_pred - y_true
    pct_greater = (diffs > 0).sum() / len(diffs) * 100
    pct_less = (diffs < 0).sum() / len(diffs) * 100
    pct_equal = (diffs == 0).sum() / len(diffs) * 100


    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'pct_greater': round(pct_greater, 2),
        'pct_less': round(pct_less, 2),
        'pct_equal': round(pct_equal, 2)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))

print(f"{len(df_success)} Daten mit {model}")
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")

    print(f"  % Greater: {values['pct_greater']}")
    print(f"  % Less: {values['pct_less']}")
    print(f"  % Equal: {values['pct_equal']}")
    print()
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

760 Daten mit Llama-3-8B-Instruct_5_shot
Metrics for AND:
  Accuracy: 0.145
  Precision: 0.126
  Recall: 0.145
  F1 Score: 0.128
  % Greater: 22.24
  % Less: 63.29
  % Equal: 14.47

Metrics for OR:
  Accuracy: 0.2
  Precision: 0.33
  Recall: 0.2
  F1 Score: 0.226
  % Greater: 60.0
  % Less: 20.0
  % Equal: 20.0

Metrics for NOT:
  Accuracy: 0.782
  Precision: 0.731
  Recall: 0.782
  F1 Score: 0.75
  % Greater: 5.53
  % Less: 16.32
  % Equal: 78.16

Metrics for DEPTH:
  Accuracy: 0.195
  Precision: 0.201
  Recall: 0.195
  F1 Score: 0.19
  % Greater: 28.29
  % Less: 52.24
  % Equal: 19.47



In [46]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows

,label_AND,model_AND
NCT,,
NCT01963754_inc,8,8
NCT00959569_inc,3,3
NCT02456532_exc,2,2
NCT02019160_exc,1,1
NCT01650792_exc,7,7
...,...,...
NCT00609531_inc,8,8
NCT02277067_exc,3,3
NCT02563535_inc,4,4
